# RT-DETRv2 R18VD — Pretrained Fine-tune (Google Colab, A100)

Restratified termal dataset ile RT-DETRv2'yi **HF pretrained agirliklardan fine-tune** eder.
From-scratch karsiligi: `colab_train_rtdetr.ipynb` (best mAP 0.4512 @ epoch 49).
Bu run scratch-vs-pretrain karsilastirmasinin eksik yarisini doldurur.

## Baslamadan once

1. `restratified_coco.zip` Drive'da olmali (`MyDrive/restratified_coco.zip`).
2. Runtime > Change runtime type > **A100 GPU** sec.

## Kesinti / resume

Checkpoint'ler ve `best.pt` dogrudan Drive'a yazilir (`MyDrive/runs_new/...`).
Oturum koptugunda: config hucresinde `RESUME_RUN`'a kesilen run'in dizin adini
ver, en son checkpoint otomatik bulunur, egitim kaldigi yerden devam eder.

In [ ]:
# GPU kontrolu — A100 bekleniyor
!nvidia-smi

In [ ]:
# Bagimliliklar. Colab'in kendi torch'u kullanilir (yeniden kurma!).
# transformers >= 4.48 gerekli (RTDetrV2 mimarisi icin).
!pip install -q -U transformers accelerate albumentations torchmetrics pycocotools openpyxl

In [ ]:
# Drive mount: dataset zip'i buradan okunur, run ciktilari buraya yazilir.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Repo'yu klonla (dataset.py, train_rtdetr.py, map_evaluator.py vs. buradan gelir).
import os

REPO_URL = "https://github.com/Yukseltt/CNN-models-optimized-to-run-on-edge-devices.git"
REPO_DIR = "/content/CNN-models-optimized-to-run-on-edge-devices"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print("Repo zaten var, guncelleniyor...")
    !git -C {REPO_DIR} pull

In [ ]:
# Dataset'i Drive'daki zip'ten LOKAL diske ac.
# ONEMLI: Drive uzerinden dogrudan okumak (35k kucuk dosya) cok yavas —
# her zaman /content altina acip oradan oku.
import os

DRIVE_ZIP = "/content/drive/MyDrive/restratified_coco.zip"
DATA_DIR  = "/content/dataset/restratified_coco"

if os.path.exists(os.path.join(DATA_DIR, "train", "_annotations.coco.json")):
    print("Dataset zaten acilmis, atlaniyor.")
else:
    assert os.path.exists(DRIVE_ZIP), f"Zip bulunamadi: {DRIVE_ZIP} — Drive'a yukledin mi?"
    os.makedirs("/content/dataset", exist_ok=True)
    print("Zip aciliyor (2.2 GB, birkac dakika surer)...")
    !unzip -q {DRIVE_ZIP} -d /content/dataset

for split in ("train", "val", "test"):
    ann = os.path.join(DATA_DIR, split, "_annotations.coco.json")
    n   = len(os.listdir(os.path.join(DATA_DIR, split, "images")))
    assert os.path.exists(ann), f"Eksik: {ann}"
    print(f"{split}: {n} goruntu")

In [ ]:
# --- KONFIGURASYON: PRETRAINED FINE-TUNE (A100 40GB) ---
import sys
from pathlib import Path

# Sadece rtdetr modul dizinini path'e ekle (repo koku HF 'datasets'i golgeler).
sys.path.insert(0, f"{REPO_DIR}/models/rtdetr_v2_r18vd")

RUNS_NEW = "/content/drive/MyDrive/runs_new"

# --- RESUME ---
# Kesilen run'in dizin ADI (None -> fresh finetune baslar).
RESUME_RUN  = None
RESUME_FROM = None
if RESUME_RUN is not None:
    _ckpts = sorted(
        Path(f"{RUNS_NEW}/{RESUME_RUN}/checkpoints").glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[1]),
    )
    assert _ckpts, f"Checkpoint bulunamadi: {RUNS_NEW}/{RESUME_RUN}/checkpoints"
    RESUME_FROM = str(_ckpts[-1])

# From-scratch run'dan (colab6) farklar — fine-tune mantigi:
#   LR 4e-4 -> 1e-4        (pretrained agirliklari bozmamak icin dusuk)
#   WARMUP 1500 -> 500     (agirliklar zaten anlamli, uzun warmup gereksiz)
#   EPOCHS 300 -> 80       (fine-tune cok daha hizli oturur)
#   PATIENCE 30 -> 15      (erken doyum beklenir)
#   FROM_SCRATCH False     (HF Hub pretrained + ignore_mismatched_sizes ile 3-sinif head)
# IMAGE_SIZE 480: scratch run ile birebir karsilastirma icin.
#   Person mAP (kucuk nesne) hedefliyorsan 640 yap; o zaman BS'leri 24'e dusur
#   (640^2 aktivasyon ~1.8x) — ama sonuc scratch ile dogrudan kiyaslanamaz olur.
CFG = {
    "EPOCHS":              80,
    "IMAGE_SIZE":          480,
    "PER_DEVICE_TRAIN_BS": 32,
    "PER_DEVICE_EVAL_BS":  32,
    "GRAD_ACCUM":          1,
    "LR":                  1e-4,
    "WARMUP_STEPS":        500,
    "WEIGHT_DECAY":        1e-4,
    "PATIENCE":            15,
    "WORKERS":             8,
    "PREFETCH_FACTOR":     4,
    "EVAL_ACCUM_STEPS":    4,
    "EVAL_STEPS":          1100,  # bs32'de ~1100 step = 1 epoch -> her epoch eval+checkpoint
    "COMPILE":             False, # RT-DETRv2 encoder'i ile uyumsuz (train_rtdetr.py notu)
    "PROJECT":             RUNS_NEW,
    "NAME":                "rtdetr_v2_r18vd_finetune_restratified_colab",
    "RESUME_FROM":         None,
    "FROM_SCRATCH":        False,  # PRETRAINED fine-tune
    "SKIP_CONFIRM":        True,
}

if RESUME_FROM is not None:
    # Resume'da run_dir checkpoint yolundan otomatik bulunur; xlsx offset'i
    # son satirdan okunur, best.pt metrigi diskten yuklenir.
    CFG["RESUME_FROM"]  = RESUME_FROM
    CFG["WARMUP_STEPS"] = 0
    print(f"RESUME modu: {RESUME_FROM}")
    print(f"Run ciktilari: {RUNS_NEW}/{RESUME_RUN}")
else:
    print("FRESH pretrained fine-tune.")
    print(f"Run ciktilari: {RUNS_NEW}/{CFG['NAME']}")

In [ ]:
# --- EGITIM ---
# A100: bf16 + tf32 otomatik aktif (train_rtdetr.py gate'liyor).
# Her EVAL_STEPS'te: eval + Excel satiri + best/last.pt + HF checkpoint (Drive'a).
# Excel/pt yazimlari Drive FUSE hatalarina dayanikli (retry + skip) — egitim olmez.
from train_rtdetr import train

train(data_dir=DATA_DIR, cfg=CFG)

## Egitim sonrasi

Tum ciktilar Drive'da: `MyDrive/runs_new/rtdetr_v2_r18vd_finetune_restratified_colab/`

- `best.pt` — en iyi `eval_map` snapshot'i
- `last.pt` — son evaluation snapshot'i
- `training_metrics.xlsx` — her eval'de bir satir
- `plots/` — egitim bitince otomatik grafikler
- `checkpoints/` — HF Trainer checkpoint'leri (resume kaynagi, son 2 tanesi)

Karsilastirma referansi (ayni dataset, 480px):
- from-scratch (colab6): best mAP 0.4512 / mAP@0.5 0.7047 @ epoch 49
- en iyi YOLO (yolov8n_cb): mAP 0.6113

Oturum kopmussa: config hucresinde `RESUME_RUN`'a run dizin adini ver,
config + egitim hucrelerini yeniden calistir.

In [ ]:
# (Opsiyonel) Resume icin son checkpoint'i bul.
!ls -lt {RUNS_NEW}/{CFG['NAME']}*/checkpoints/ 2>/dev/null | head -20